In [ ]:
#this script keep only the points that fall within Nigerian territory and saves the result. create a new file with division in layer based on the manual check classification.

In [ ]:
# Create new file with division in layer based on the manual check classification
# Process one layer at a time to avoid memory issues

import geopandas as gpd
import pandas as pd

DATA_DIR = "dataset_big"

points_file = f"{DATA_DIR}/lpg_infrastructure_3857.gpkg"
boundary_file = f"{DATA_DIR}/Country_boundaries.geojson"
output_file = f"{DATA_DIR}/manual_check_nig_3857.gpkg"

# Load boundary once
boundary = gpd.read_file(boundary_file)
nigeria = boundary.geometry.union_all()

# Classification mapping
type_map = {
    "primary depot": "primary_depot",
    "filling plant": "filling",
    "retailer": "reseller",
    "invalid": "false_points"
}

# For each verified type, collect points from all original layers
for vt, layer_name in type_map.items():
    all_points = []
    for orig_layer in ["both", "depot_only", "filling_only"]:
        try:
            gdf = gpd.read_file(points_file, layer=orig_layer)
            # Filter by verified_type
            mask = gdf["verified_type"] == vt
            gdf_filtered = gdf[mask].copy()
            if not gdf_filtered.empty:
                all_points.append(gdf_filtered)
        except Exception as e:
            print(f"Warning: could not read layer {orig_layer}: {e}")
            continue

    if not all_points:
        print(f"No points found for {vt}")
        continue

    combined = pd.concat(all_points, ignore_index=True)
    print(f"{vt}: {len(combined)} points before spatial filter")

    # Reproject if needed
    if combined.crs != boundary.crs:
        combined = combined.to_crs(boundary.crs)

    # Keep only points within Nigeria
    combined_nigeria = combined[combined.geometry.within(nigeria)]
    print(f"  After Nigeria filter: {len(combined_nigeria)} points")

    if not combined_nigeria.empty:
        combined_nigeria = combined_nigeria.to_crs("EPSG:3857")
        combined_nigeria.to_file(output_file, layer=layer_name, driver="GPKG")
        print(f"  Saved to layer '{layer_name}'")

# Check for any points with verified_type not in type_map (e.g., 'both')
unclassified = []
for orig_layer in ["both", "depot_only", "filling_only"]:
    try:
        gdf = gpd.read_file(points_file, layer=orig_layer)
        mask = ~gdf["verified_type"].isin(type_map.keys())
        if mask.any():
            unclassified.append(gdf[mask].copy())
    except:
        continue

if unclassified:
    unclassified_all = pd.concat(unclassified, ignore_index=True)
    # Filter to Nigeria
    if unclassified_all.crs != boundary.crs:
        unclassified_all = unclassified_all.to_crs(boundary.crs)
    unclassified_nigeria = unclassified_all[unclassified_all.geometry.within(nigeria)]
    if not unclassified_nigeria.empty:
        unclassified_nigeria = unclassified_nigeria.to_crs("EPSG:3857")
        unclassified_nigeria.to_file(output_file, layer="unclassified", driver="GPKG")
        print(f"Saved {len(unclassified_nigeria)} unclassified points to layer 'unclassified'")

print(f"\nDone. Output saved to {output_file}")

primary depot: 14 points before spatial filter
  After Nigeria filter: 14 points
  Saved to layer 'primary_depot'
filling plant: 376 points before spatial filter
  After Nigeria filter: 376 points
  Saved to layer 'filling'
retailer: 90 points before spatial filter
  After Nigeria filter: 89 points
  Saved to layer 'reseller'
invalid: 190 points before spatial filter
  After Nigeria filter: 187 points
  Saved to layer 'false_points'

Done. Output saved to manual_check_nig_3857.gpkg
